<a href="https://colab.research.google.com/github/Potdooshami/2H_TaSe2_Tc_STM/blob/main/PYMAIA6_day3_CNN_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 데이터 불러오기

In [ ]:
!pip install torchinfo
!pip install torchmetrics matplotlib
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torchinfo import summary
# 1. 데이터 불러오기
# For displaying, we might want the raw images, so loading again without normalize for visualization
raw_train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
raw_test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())

# Extract data for shape printing and visualization
x_train_raw = raw_train_dataset.data.numpy()
y_train_raw = raw_train_dataset.targets.numpy()
x_test_raw = raw_test_dataset.data.numpy()
y_test_raw = raw_test_dataset.targets.numpy()

print("x_train shape:", x_train_raw.shape)
print("y_train shape:", y_train_raw.shape)
print("x_test shape:", x_test_raw.shape)
print("y_test shape:", y_test_raw.shape)

# 2. 임의의 100개 데이터 인덱스 선택
indices = np.random.choice(len(x_train_raw), size=100, replace=False)

# 3. 10x10 격자로 이미지 출력
fig, axes = plt.subplots(10, 10, figsize=(10, 10))
for i, index in enumerate(indices):
    row = i // 10
    col = i % 10
    # PyTorch datasets return PIL images or tensors, need to convert to numpy for imshow
    # If using raw_train_dataset.data, it's already a numpy array
    axes[row, col].imshow(x_train_raw[index], cmap='gray')
    axes[row, col].axis('off')
plt.suptitle("MNIST Training Data (100 samples) - PyTorch", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
x_pxl = 9# @param {type:"raw"}
y_pxl = 20# @param {type:"raw"}
index_train = 10# @param {type:"raw"}
plt.imshow(x_train_raw[index_train], cmap='gray')
plt.xlabel("x pixel")
plt.ylabel("y pixel")
plt.plot(x_pxl,y_pxl,'ro')
plt.title(f"Label:{y_train_raw[index_train]} index_train: {index_train}")
print(f'The value of (x={x_pxl},y={y_pxl}) is',
      x_train_raw[index_train][y_pxl,x_pxl])

# 신경망 만들기(클래스 선언)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader,random_split
from tqdm import tqdm


# 4. MLP 모델 정의 (PyTorch)
class Perceptron(nn.Module):
    def __init__(self):
        super(Perceptron, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 10) # Input: 784, Output: 128
    def forward(self, x):
        x = x.view(-1,28*28)
        x = self.fc1(x)
        return x # Logits
class MLP_naive(nn.Module):
    def __init__(self):
        super(MLP_naive, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 128) # Input: 784, Output: 128
        self.fc2 = nn.Linear(128, 64)      # Input: 128, Output: 64
        self.fc3 = nn.Linear(64, 10)       # Input: 64, Output: 10 (for 0-9 classes)

    def forward(self, x):
        x = x.view(-1,28*28)
        x = self.fc1(x)
        x = self.fc2(x)
        x = self.fc3(x)
        return x # Logits
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(28 * 28, 128) # Input: 784, Output: 128
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(128, 64)      # Input: 128, Output: 64
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(64, 10)       # Input: 64, Output: 10 (for 0-9 classes)

    def forward(self, x):
        x = x.view(-1,28*28)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        return x # Logits
class FlexibleMLPSequential(nn.Module):
    def __init__(self, layer_sizes):
        super(FlexibleMLPSequential, self).__init__()

        # Build a list of layers first
        modules = []
        for i in range(len(layer_sizes) - 1):
            modules.append(nn.Linear(layer_sizes[i], layer_sizes[i+1]))
            if i < len(layer_sizes) - 2:
                modules.append(nn.ReLU())

        # Unpack the list into nn.Sequential
        # This registers everything at once and defines the forward pass order
        self.network = nn.Sequential(*modules)

    def forward(self, x):
        # Flatten the input automatically based on the first layer's input features
        # self.network[0] points to the very first nn.Linear layer
        x = x.view(-1, self.network[0].in_features)

        # No need for loops in forward; nn.Sequential handles it internally
        return self.network(x)
class HighPerfCNN(nn.Module):
    def __init__(self):
        super(HighPerfCNN, self).__init__()

        # Input shape: (batch_size, 1, 28, 28)
        # First convolutional block
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.relu1 = nn.ReLU()

        # Second convolutional block
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.relu2 = nn.ReLU()

        # Max pooling downsamples 28x28 to 14x14
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.dropout1 = nn.Dropout(0.25)

        # Third convolutional block
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.relu3 = nn.ReLU()

        # Max pooling downsamples 14x14 to 7x7
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.dropout2 = nn.Dropout(0.25)

        # Fully connected layers
        # After two pools, the feature map size is 128 channels * 7 * 7
        self.fc1 = nn.Linear(128 * 7 * 7, 256)
        self.bn4 = nn.BatchNorm1d(256)
        self.relu4 = nn.ReLU()
        self.dropout3 = nn.Dropout(0.5)

        # Final output layer (10 classes for digits 0-9)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        # Feature extraction
        x = self.relu1(self.bn1(self.conv1(x)))
        x = self.relu2(self.bn2(self.conv2(x)))
        x = self.dropout1(self.pool1(x))

        x = self.relu3(self.bn3(self.conv3(x)))
        x = self.dropout2(self.pool2(x))

        # Flatten the feature maps for the dense layers
        x = x.view(-1, 128 * 7 * 7)

        # Classification
        x = self.dropout3(self.relu4(self.bn4(self.fc1(x))))
        x = self.fc2(x)

        return x  # Logits
import torch
import torch.nn as nn

class LowParamCNN(nn.Module):
    def __init__(self):
        super(LowParamCNN, self).__init__()

        # Input shape: (batch_size, 1, 28, 28)
        # First convolutional block: reduce channels to 16
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu1 = nn.ReLU()

        # Second convolutional block: reduce channels to 32
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.relu2 = nn.ReLU()

        # Max pooling downsamples 28x28 to 14x14
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.dropout1 = nn.Dropout(0.2)

        # Third convolutional block: reduce channels to 64
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.relu3 = nn.ReLU()

        # Max pooling downsamples 14x14 to 7x7
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.dropout2 = nn.Dropout(0.2)

        # Global Average Pooling: collapses (batch, 64, 7, 7) to (batch, 64, 1, 1)
        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        # Final output layer: direct mapping from 64 channels to 10 classes
        # This drastically removes millions of parameters from the previous FC layers
        self.fc = nn.Linear(64, 10)

    def forward(self, x):
        x = self.relu1(self.bn1(self.conv1(x)))
        x = self.relu2(self.bn2(self.conv2(x)))
        x = self.dropout1(self.pool1(x))

        x = self.relu3(self.bn3(self.conv3(x)))
        x = self.dropout2(self.pool2(x))

        # Apply GAP and flatten for the single linear layer
        x = self.gap(x)
        x = torch.flatten(x, 1)

        x = self.fc(x)
        return x


In [ ]:
import torchmetrics
model_dict = dict()
device_gpu = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device_cpu = torch.device("cpu")
device = device_gpu
train_size = 50000
val_size = 10000
generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(raw_train_dataset, [train_size, val_size], generator=generator)
loss_fn = nn.CrossEntropyLoss();print('',loss_fn)

batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=10000, shuffle=False)
test_loader = DataLoader(raw_test_dataset, batch_size=10000, shuffle=False)
def test(model, loss_fn, test_loader, device):
    model.eval()  # Turn off layers like dropout or batchnorm for evaluation
    running_loss = 0.0
    correct_predictions = 0

    with torch.no_grad():
        for i, (inputs, labels) in enumerate(test_loader):
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            loss = loss_fn(outputs, labels)

            # Multiply by inputs.size(0) to get the total loss of the current batch
            running_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            correct_predictions += (predicted == labels).sum().item()

        # Divide total accumulated loss by the total number of samples
        avg_test_loss = running_loss / len(test_loader)
        test_accuracy = correct_predictions / len(test_loader.dataset)

    return avg_test_loss, test_accuracy
def train(model,loss_fn,train_loader,device,optimizer,num_epochs,val_loader):
  print('inside the function')
  history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}
  for epoch in range(num_epochs):
    if epoch == 0:
      print('in 1st loop')
    model.train()
    running_loss = 0.0 #배치마다 나온 로스값을 누적하기 위한 플레이스 홀더. avg_train_loss의 발사대
    correct_predictions = 0 #Accuracy 측정을 위해
    progress_bar = tqdm(
            train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}]", leave=True
        )
    for i, (inputs, labels) in enumerate(progress_bar):
      # if i == 0:
      #   print('im in 1st batch')
      optimizer.zero_grad() #이전 배치에서의 경사를 초기화
      inputs = inputs.to(device)
      labels = labels.to(device)
      outputs = model(inputs)
      loss = loss_fn(outputs, labels)
      loss.backward()
      optimizer.step()
      running_loss += loss.item()
      _, predicted = torch.max(outputs.data, 1)
      correct_predictions += (predicted == labels).sum().item()
      progress_bar.set_postfix({"batch_loss": f"{loss.item():.4f}"})

    avg_train_loss = running_loss / len(train_loader)
    train_accuracy = correct_predictions / len(train_loader.dataset)
    history['loss'].append(avg_train_loss)
    history['accuracy'].append(train_accuracy)
    val_loss,val_accuracy = test(model,loss_fn,val_loader,device)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_accuracy)
    progress_bar.set_postfix({"batch_loss": f"{loss.item():.4f}"})
    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Train Acc: {train_accuracy*100:.1f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy*100:.1f}%"
    )
  return history
def show_history(history):
  plt.figure(figsize=(10, 5))
  plt.plot(history['loss'], label='Train Loss')
  plt.plot(history['val_loss'], label='Val Loss')
  plt.xlabel('Epoch')
  plt.ylabel('Loss')
  plt.title('Loss Curve (PyTorch)')
  plt.legend()
  plt.grid(True)
  plt.show()

  plt.figure(figsize=(10, 5))
  plt.plot(history['accuracy'], label='Train Accuracy')
  plt.plot(history['val_accuracy'], label='Val Accuracy')
  plt.xlabel('Epoch')
  plt.ylabel('Accuracy')
  plt.title('Accuracy Curve (PyTorch)')
  plt.legend()
  plt.grid(True)
  plt.show()
def print_test(model):
  X_test,target = next(iter(test_loader))
  X_test = X_test.to(device)
  target = target.to(device)
  with torch.no_grad():
    foo = model.forward(X_test)
    _,pred = foo.max(dim=1)
  metric = torchmetrics.ConfusionMatrix(task="multiclass", num_classes=10).to(device)
  cm = metric(pred, target)
  metric.plot()
  accuracy = torch.trace(cm) / cm.sum()
  plt.title(f'Accuracy: {(100*accuracy.item()):.3f}%')
  model_dict[model.__class__.__name__] = model
  return cm






## Model1: Perceptron

In [ ]:
model = LowParamCNN() # @param ["Perceptron()", "MLP()", "MLP_naive()", "FlexibleMLPSequential([28*28,10])","HighPerfCNN()","LowParamCNN()"]{allow-input: true,type:"raw"}
model.to(device)
optimizer = optim.Adam(model.parameters());

print(summary(model,(1,1,28,28),verbose=0,col_names=["input_size", "output_size", "num_params", "mult_adds", "trainable"]))

num_epochs = 20

In [ ]:
hisory = train(model,loss_fn,train_loader,device,optimizer,10,val_loader)
show_history(hisory)
print_test(model)

In [ ]:


X_test,target = next(iter(test_loader))
print(X_test.shape)
ss = model.forward(X_test)
print(ss.shape)
_,pred = ss.max(dim=1)


# 증강 데이터 테스트

In [ ]:
def html_canvas():
  from IPython.display import HTML, display
  import base64

  from google.colab import output

  # HTML + JavaScript 코드
  canvas_html = """
  <canvas id="canvas" width="280" height="280" style="border:1px solid black;"></canvas><br>
  <button onclick="saveCanvas()">Save as PNG</button>
  <button onclick="clearCanvas()">Clear Canvas</button>
  <script>
  let canvas = document.getElementById('canvas');
  let ctx = canvas.getContext('2d');
  let drawing = false;

  canvas.addEventListener('mousedown', (e) => { drawing = true; ctx.beginPath(); });
  canvas.addEventListener('mouseup', () => { drawing = false; });
  canvas.addEventListener('mousemove', (e) => {
      if (!drawing) return;
      ctx.lineWidth = 10;
      ctx.lineCap = 'round';
      ctx.strokeStyle = 'black';
      let rect = canvas.getBoundingClientRect();
      ctx.lineTo(e.clientX - rect.left, e.clientY - rect.top);
      ctx.stroke();
  });

  function saveCanvas() {
      let dataURL = canvas.toDataURL('image/png');
      google.colab.kernel.invokeFunction('notebook.save_canvas', [dataURL], {});
  }

  function clearCanvas() {
      ctx.clearRect(0, 0, canvas.width, canvas.height);
  }
  </script>
  """

  # 데이터를 저장하는 Python 콜백 함수
  def save_canvas(data):
      png_data = base64.b64decode(data.split(',')[1])
      with open('canvas.png', 'wb') as f:
          f.write(png_data)
      print("Saved as canvas.png")


  # Colab 콜백 함수 등록
  output.register_callback('notebook.save_canvas', save_canvas)

  # HTML 렌더링
  display(HTML(canvas_html))
def get_tensor_from_png():
  ''' colab 구글드라이브에 저장된 png를 텐서로 바꿈
  Args:
  Returns:
    tensor: (1,28,28) size
  '''
  from PIL import Image
  image = Image.open('canvas.png').convert('RGBA')
  image = np.array(image.resize((28, 28))).astype('float32') / 255.0
  image = image[:, :, 3]  # 알파 채널
  transform = transforms.ToTensor()
  tensor = transform(image)
  tensor = tensor.reshape(1,28,28)
  return tensor
def get_tensor_agumented_test(ind_train=42,xpxl=0,ypxl=0):
  '''translation을 통해 test 데이터를 살짝 변형시킨 인풋 생성
  Args:
    ind_train (int):
    xpxl (int):
    ypxl (int):
  Returns:
    tensor (tensor): (1,28,28) size
    label (int):
  '''
  tensor,label=raw_test_dataset[ind_train]
  tensor = torch.roll(tensor, shifts=(xpxl, ypxl), dims=(1, 2))
  return tensor, label
def vis_tensor(tensor):
  '''tensor를 plt로 시각화
  Args:
    tensor (tensor): (1,28,28) size
  Returns:
  '''
  arr = tensor.detach().cpu().numpy()
  arr = arr.reshape(28,28)
  plt.imshow(arr)
def get_predict(model,tensor):
  ''' model의 아웃풋 출력
  Args:
    model (model): forward를 가지고 있음
    tensor (tensor): (1,28,28) size
  Returns:
    predict (array): (10,)
  '''
  tensor = tensor.to(device)
  tensor = tensor.view(1,1,28,28)
  with torch.no_grad():
    x_forward = model.forward(tensor)
    predict = torch.softmax(x_forward, dim=-1).detach().cpu().numpy()
    predict = predict.squeeze()
  return predict
def vis_predict(predict):
  ''' visualize predict
  Args:
    predict (array): (10,)
  Returns:
  '''
  plt.bar(np.arange(10),predict)
  plt.xlabel('labels')
  plt.ylabel('probability')
  plt.xticks(np.arange(10))


def compare_model(model_1,model_2,tensor):
  '''compare two model with tensor
  Args:
    model1 (model): forward를 가지고 있음
    model2 (model): forward를 가지고 있음
    tensor (tensor): (1,28,28) size
  Returns:
  '''
  models = [model_1,model_2]
  fig,axs = plt.subplots(1,3,figsize=(9,3))
  plt.sca(axs[0])
  vis_tensor(tensor)
  for ind in range(2):
    model = models[ind]
    plt.sca(axs[ind+1])
    vis_predict(get_predict(model,tensor))
    plt.title(model.__class__.__name__)
  plt.tight_layout()



In [ ]:
ind_train=550 # @param
xpxl=0# @param
ypxl=-6# @param
model_1 ="MLP" # @param ["Perceptron", "MLP", "MLP_naive", "FlexibleMLPSequential","HighPerfCNN","LowParamCNN"]
model_2 ="LowParamCNN" # @param ["Perceptron", "MLP", "MLP_naive", "FlexibleMLPSequential","HighPerfCNN","LowParamCNN"]
x,_ = get_tensor_agumented_test(ind_train,xpxl,ypxl)
compare_model(model_dict[model_1],model_dict[model_2],x)

In [ ]:
html_canvas()

# x,label=raw_test_dataset[1]
print(x.shape)


In [ ]:
model_1 ="HighPerfCNN" # @param ["Perceptron", "MLP", "MLP_naive", "FlexibleMLPSequential","HighPerfCNN","LowParamCNN"]
model_2 ="FlexibleMLPSequential" # @param ["Perceptron", "MLP", "MLP_naive", "FlexibleMLPSequential","HighPerfCNN","LowParamCNN"]
compare_model(model_dict[model_1],model_dict[model_2],get_tensor_from_png())